### Prérequis

- Avoir un compte IBKR
- IB GATEWAY ouvert et connecté au compte IBKR
  - ❌ Read-Only API décoché
  - Port : `4002' ?

# Imports et configuration

In [ ]:
from ib_insync import *
import nest_asyncio
nest_asyncio.apply()

import pandas as pd
import plotly.graph_objects as go
from datetime import datetime, timedelta
import time
import os

In [ ]:
# Configuration initiale
nest_asyncio.apply()
DATA_DIR = './data/sector_etfs_daily'
os.makedirs(DATA_DIR, exist_ok=True)

SECTOR_ETFS = {
    'XLK': 'Technology', 'XLF': 'Financials', 'XLE': 'Energy',
    'XLV': 'Health Care', 'XLI': 'Industrials', 'XLC': 'Communication Services',
    'XLY': 'Consumer Discretionary', 'XLP': 'Consumer Staples',
    'XLU': 'Utilities', 'XLRE': 'Real Estate', 'XLB': 'Materials'
}

# Creation connecteur
ib = IB()

# Connexion en mode lecture seule (sécurisé)
ib.connect('127.0.0.1', 4002, clientId=1, readonly=True)

print(f"Début du téléchargement pour {len(SECTOR_ETFS)} secteurs...")

for ticker, sector_name in SECTOR_ETFS.items():
    # 1. Création et qualification du contrat
    contract = Stock(ticker, 'SMART', 'USD')

    # Test si l'etf existe
    ib.qualifyContracts(contract)
    
    # 2. Requête One-Shot (6 ans pour couvrir 2020-2026)
    # Pour du '1 day', IB accepte plusieurs années sans broncher :) 
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='', 
        durationStr='6 Y', 
        barSizeSetting='1 day', 
        whatToShow='TRADES',
        useRTH=True
    )
    
    if bars:
        df = util.df(bars)

        # Ajout des infos de secteur
        df['ticker'] = ticker
        df['sector'] = sector_name
        
        # 3. Sauvegarde en CSV
        file_path = os.path.join(DATA_DIR, f"{ticker}.csv")
        df.to_csv(file_path, index=False)
        print(f"✓ {ticker} ({sector_name}) : {len(df)} jours sauvegardés.")
    else:
        print(f"✗ Erreur pour {ticker}")

print("\nTerminé ! Tes fichiers sont dans :", DATA_DIR)

In [ ]:
ib.disconnect()